# Exploración del GeoJSON de barrios y veredas de Medellín

Este notebook analiza `data/limite_barrio_vereda_cata.geojson` y mide cuántos barrios declarados por los estudiantes
se pueden relacionar con un polígono oficial. Sus cifras respaldan las decisiones del cambio `ubicacion-por-niveles`.

**Cómo ejecutarlo:** `conda activate medellin-analysis312 && jupyter lab` desde la raíz del repositorio, y *Run All*.

**Datos personales:** las secciones 5 a 8 leen `original/` y `coordenadas/coordenadas.xlsx` en local. Antes de cada
commit hay que limpiar las salidas (`jupyter nbconvert --clear-output --inplace notebooks/exploracion_geojson.ipynb`);
las cifras que deban conservarse se copian, solo agregadas, a `notebooks/hallazgos_geojson.md`.

In [ ]:
# Comprobación del entorno: la reproyección necesita la base de datos de PROJ.
import os

import pyproj

datos_proj = pyproj.datadir.get_data_dir()
if not datos_proj or not os.path.exists(os.path.join(datos_proj, "proj.db")):
    raise RuntimeError(
        "pyproj no encuentra la base de datos de PROJ (proj.db). "
        "Abre Jupyter con el entorno activado:\n"
        "  conda activate medellin-analysis312 && jupyter lab"
    )
pyproj.Transformer.from_crs("EPSG:9377", "EPSG:4326", always_xy=True)
print("Base de datos de PROJ:", datos_proj)

In [ ]:
import difflib
import re
import unicodedata
from collections import Counter
from pathlib import Path

import geopandas as gpd
import pandas as pd

# El notebook puede ejecutarse desde notebooks/ (Jupyter, nbconvert) o desde la raiz del repositorio.
RAIZ = next(
    ruta for ruta in (Path.cwd(), *Path.cwd().parents)
    if (ruta / "data" / "limite_barrio_vereda_cata.geojson").exists()
)
RUTA_GEOJSON = RAIZ / "data" / "limite_barrio_vereda_cata.geojson"
RUTA_ALIAS = RAIZ / "data" / "alias_barrios.csv"
RUTA_ORIGINAL = RAIZ / "original"
RUTA_COORDENADAS = RAIZ / "coordenadas" / "coordenadas.xlsx"

CRS_METRICO = "EPSG:9377"  # MAGNA-SIRGAS Origen Nacional, en metros
CRS_WGS84 = "EPSG:4326"
CENTROIDE_MEDELLIN = (6.25184, -75.56359)  # punto al que OpenCage lleva las consultas que solo resuelve por ciudad

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)

## 1. Esquema, sistema de coordenadas, cobertura y validez

¿Qué contiene el archivo, en qué sistema de coordenadas está y se puede usar sin reparar geometrías?

In [ ]:
barrios = gpd.read_file(RUTA_GEOJSON)
# codigo es el identificador estable: se trata siempre como texto para conservar los ceros (0806).
barrios["codigo"] = barrios["codigo"].astype(str)

esquema = pd.DataFrame({
    "tipo": barrios.drop(columns="geometry").dtypes.astype(str),
    "nulos": barrios.drop(columns="geometry").isna().sum(),
    "ejemplo": barrios.drop(columns="geometry").iloc[0],
})
esquema

In [ ]:
fechas_sincronizacion = pd.to_datetime(barrios["fecha_sincronizacion"], unit="ms").dt.date.unique()
lon_min, lat_min, lon_max, lat_max = barrios.to_crs(CRS_WGS84).total_bounds

print("Sistema de coordenadas:", barrios.crs)
print("Polígonos:", len(barrios))
print("Tipos de geometría:", barrios.geom_type.value_counts().to_dict())
print("Geometrías inválidas:", int((~barrios.is_valid).sum()))
print("Comunas y corregimientos:", barrios["nombre_comuna"].nunique())
print("Fecha de sincronización:", list(fechas_sincronizacion))
print(f"Extensión en WGS84: latitud {lat_min:.4f} a {lat_max:.4f}, longitud {lon_min:.4f} a {lon_max:.4f}")
print("Códigos únicos:", barrios["codigo"].is_unique,
      "| longitudes:", barrios["codigo"].str.len().value_counts().to_dict(),
      "| con cero a la izquierda:", int(barrios["codigo"].str.startswith("0").sum()))

assert len(barrios) == 349
assert int((~barrios.is_valid).sum()) == 0
assert barrios["codigo"].is_unique
assert barrios["codigo"].map(type).eq(str).all() and barrios["codigo"].str.startswith("0").any()

In [ ]:
barrios.groupby("nombre_comuna").agg(
    poligonos=("codigo", "size"),
    urbanos=("indicador_ur", lambda s: int((s == "U").sum())),
    rurales=("indicador_ur", lambda s: int((s == "R").sum())),
).sort_values("poligonos", ascending=False)

**Conclusión.** El archivo tiene 349 polígonos válidos en EPSG:9377 (metros), con 21 comunas y corregimientos de
Medellín y una única fecha de sincronización. `codigo` es único y conserva los ceros a la izquierda si se lee como
texto. Solo cubre Medellín: los demás municipios del área metropolitana necesitan otra fuente o un punto fijo por
municipio.

## 2. Normalización de nombres

Convención (especificación `referencia-barrios`): mayúsculas, sin diacríticos (`Ñ` → `N`), puntuación sustituida por
espacios, espacios colapsados y extremos recortados. ¿Hay nombres oficiales distintos que colisionan al normalizarlos?

In [ ]:
def normalizar(texto):
    """Normaliza un nombre para compararlo: mayúsculas, sin diacríticos ni puntuación y con espacios simples."""
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return ""
    descompuesto = unicodedata.normalize("NFKD", str(texto).upper())
    sin_diacriticos = "".join(c for c in descompuesto if not unicodedata.combining(c))
    sin_puntuacion = re.sub(r"[^\w\s]|_", " ", sin_diacriticos)
    return re.sub(r"\s+", " ", sin_puntuacion).strip()


# Escenarios de la especificación
assert normalizar("Área de Expansión Pajarito") == "AREA DE EXPANSION PAJARITO"
assert normalizar("La Cabaña") == "LA CABANA"
assert normalizar("  carlos e.  restrepo ") == "CARLOS E RESTREPO"

barrios["nombre_norm"] = barrios["nombre_barrio"].map(normalizar)
variantes = barrios.groupby("nombre_norm")["nombre_barrio"].agg(lambda s: sorted({repr(v) for v in s}))
colisiones = variantes[variantes.map(len) > 1]
print("Nombres oficiales distintos:", barrios["nombre_barrio"].nunique(),
      "| tras normalizar:", barrios["nombre_norm"].nunique())
colisiones.to_frame("variantes originales")

**Conclusión.** La normalización solo hace colisionar nombres que ya eran el mismo barrio escrito con un espacio final
(`'El Salado'` y `'El Salado '`). Esa colisión no pierde información, pero añade un nombre repetido más a la sección 3.

## 3. Nombres repetidos, zonas `U`/`R` y polígonos sin indicador

¿Qué nombres corresponden a más de un polígono y la zona `URBANO`/`RURAL` de la dirección basta para distinguirlos?

In [ ]:
repetidos_originales = barrios["nombre_barrio"].value_counts()
repetidos_originales = repetidos_originales[repetidos_originales > 1]

conteo_norm = barrios["nombre_norm"].value_counts()
nombres_repetidos = conteo_norm[conteo_norm > 1].index
repetidos = (
    barrios[barrios["nombre_norm"].isin(nombres_repetidos)]
    .sort_values(["nombre_norm", "indicador_ur"])
    [["nombre_norm", "codigo", "nombre_comuna", "indicador_ur"]]
)
zonas_por_nombre = repetidos.groupby("nombre_norm")["indicador_ur"].agg(lambda s: "".join(sorted(s.fillna("?"))))

print("Nombres repetidos en el campo original:", len(repetidos_originales))
print("Nombres repetidos tras normalizar:", len(nombres_repetidos), "en", len(repetidos), "polígonos")
print("Combinaciones de zona:", zonas_por_nombre.value_counts().to_dict())
assert len(repetidos_originales) == 8
repetidos

In [ ]:
sin_indicador = barrios[barrios["indicador_ur"].isna()][["codigo", "nombre_barrio", "nombre_comuna"]]
print("Polígonos sin indicador_ur:", len(sin_indicador))
assert len(sin_indicador) == 4
sin_indicador

**Conclusión.** El campo original tiene 8 nombres repetidos; al normalizar aparece un noveno (`EL SALADO`). En todos
menos uno, un polígono es urbano y el otro rural, así que la zona de la dirección los resuelve. La excepción es
`SUBURBANO EL LLANO`, con dos polígonos rurales en corregimientos distintos, que queda sin resolver por zona. Los 4
polígonos sin indicador son las "Área de Expansión"; ninguno tiene un nombre repetido, así que no afectan a la regla.

## 4. Punto representativo de cada polígono

¿Qué punto usar como "punto cero del barrio"? Se comparan `centroid` y `representative_point()`, calculados en
EPSG:9377 (metros) y luego transformados a WGS84.

In [ ]:
centroides = barrios.geometry.centroid
representativos = barrios.geometry.representative_point()
centroide_fuera = ~centroides.within(barrios.geometry)
representativo_fuera = ~representativos.within(barrios.geometry)
separacion = centroides.distance(representativos)

print("Centroides fuera de su polígono:", int(centroide_fuera.sum()))
print("Puntos representativos fuera de su polígono:", int(representativo_fuera.sum()))
print("Separación centroide-representativo (m): mediana",
      round(float(separacion.median())), "| máxima", round(float(separacion.max())))
assert int(representativo_fuera.sum()) == 0

barrios.loc[centroide_fuera, ["codigo", "nombre_barrio", "nombre_comuna"]].assign(
    separacion_m=separacion[centroide_fuera].round()
)

In [ ]:
# Tabla del punto cero de cada barrio en WGS84, que usará el nivel BARRIO de ubicacion-por-niveles.
punto_wgs84 = gpd.GeoSeries(representativos, crs=CRS_METRICO).to_crs(CRS_WGS84)
puntos_barrio = barrios[["codigo", "nombre_barrio", "nombre_comuna", "indicador_ur"]].assign(
    latitud=punto_wgs84.y.round(6), longitud=punto_wgs84.x.round(6)
)
puntos_barrio.head()

**Conclusión.** Algunos centroides caen fuera de su polígono (barrios con forma de media luna o multipolígonos), mientras
que `representative_point()` siempre cae dentro. El punto cero del barrio debe ser el punto representativo, calculado en
EPSG:9377 y transformado a WGS84 solo al final.

## 5. Extracción del barrio declarado

Heurística exploratoria (diseño, D3): se toma el texto anterior a `-*-`, se quitan la vía y los números iniciales y el
barrio declarado es el **sufijo más largo** del resto que coincide, normalizado, con un nombre oficial o un alias. Lo que
queda delante es el complemento. Se parte del texto original, no de `estandarizado/`, porque la estandarización actual
corrompe nombres (`BUENOS` → `BUE#S`).

In [ ]:
# Se ignoran los archivos de bloqueo que crean LibreOffice (.~lock.*#) y Excel (~$*) mientras el archivo está abierto.
archivos_originales = [
    ruta for ruta in RUTA_ORIGINAL.glob("*") if ruta.is_file() and not ruta.name.startswith((".", "~$"))
]
assert len(archivos_originales) == 1, "original/ debe contener exactamente un archivo, igual que exige main.py"
registros = pd.read_excel(archivos_originales[0])
medellin = registros[registros["Ciudad Residencia"] == "MEDELLIN - ANTIOQUIA"].copy()

PALABRAS_VIA = {
    "CL", "CLL", "CALLE", "CR", "CRA", "KR", "KRA", "CRR", "CARRERA", "TV", "TR", "TRANSVERSAL",
    "DG", "DIAGONAL", "AV", "AVENIDA", "CIRCULAR", "CQ", "CIR", "AUTOPISTA",
}
# Tras la vía: números, letras sueltas y sufijos de cuadrante (45, 43A, B, BIS, SUR, NO...).
TOKEN_NUMERACION = re.compile(r"^(\d+[A-Z]?|[A-Z]|BIS|SUR|NORTE|ESTE|OESTE|OCCIDENTE|NO|NRO|NUM|NUMERO)$")
NUMERO = re.compile(r"^\d+[A-Z]?$")
PALABRAS_VIVIENDA = {
    "CASA", "APTO", "APT", "AP", "APARTAMENTO", "UNIDAD", "EDIFICIO", "ED", "URBANIZACION", "URB",
    "TORRE", "BLOQUE", "BL", "INTERIOR", "INT", "PISO", "ETAPA", "CONJUNTO", "MZ", "MANZANA", "LOTE", "LOCAL",
}


def separar_direccion(texto):
    """Devuelve (tokens, zona): los tokens normalizados entre la vía y '-*-', y la zona URBANO/RURAL o None."""
    texto = "" if pd.isna(texto) else str(texto)
    antes, separador, despues = texto.partition("-*-")
    zona = normalizar(despues) if separador else ""
    tokens = normalizar(antes).split()
    if tokens and tokens[0] in PALABRAS_VIA:
        tokens = tokens[1:]
    while tokens and TOKEN_NUMERACION.match(tokens[0]):
        tokens = tokens[1:]
    return tokens, (zona if zona in ("URBANO", "RURAL") else None)


def sufijo_mas_largo(tokens, nombres, max_tokens=8):
    """Devuelve (nombre, complemento): el sufijo más largo de tokens que está en nombres, y los tokens previos."""
    for n in range(min(len(tokens), max_tokens), 0, -1):
        candidato = " ".join(tokens[-n:])
        if candidato in nombres:
            return candidato, tokens[:-n]
    return None, tokens


def buscar_nombre(tokens, nombres):
    """Como sufijo_mas_largo, pero si no hay coincidencia reintenta sin los números finales (FATIMA 4)."""
    nombre, complemento = sufijo_mas_largo(tokens, nombres)
    if nombre is None:
        recortados = list(tokens)
        while recortados and NUMERO.match(recortados[-1]):
            recortados.pop()
        if len(recortados) < len(tokens):
            nombre, complemento = sufijo_mas_largo(recortados, nombres)
    return nombre, complemento


# Nombres que solo existen como comuna o corregimiento (ningún barrio se llama así). Se buscan junto con los barrios
# para que, por ejemplo, SAN ANTONIO DE PRADO no se asigne al barrio PRADO por su sufijo.
nombres_oficiales = set(barrios["nombre_norm"])
comunas_sin_barrio = {normalizar(n) for n in barrios["nombre_comuna"].unique()} - nombres_oficiales

partes = medellin["Direccion de residencia"].map(separar_direccion)
medellin["tokens"] = partes.str[0]
medellin["zona"] = partes.str[1]
con_texto = medellin["tokens"].map(len) > 0

print("Registros en el archivo original:", len(registros))
print("Registros de Medellín:", len(medellin))
print("Con zona URBANO/RURAL tras '-*-':", int(medellin["zona"].notna().sum()), medellin["zona"].value_counts().to_dict())
print("Con texto entre la vía y '-*-':", int(con_texto.sum()))

In [ ]:
coincidencia_oficial = medellin["tokens"].map(lambda t: buscar_nombre(t, nombres_oficiales | comunas_sin_barrio))
es_comuna = coincidencia_oficial.str[0].isin(comunas_sin_barrio)
medellin["barrio_oficial"] = coincidencia_oficial.str[0].where(~es_comuna)
medellin["complemento"] = coincidencia_oficial.str[1].map(" ".join)

# Sufijos cuyo token anterior no es una palabra de vivienda ni numeración: pueden ser parte de otro nombre
# (por ejemplo, un texto que acaba en CORDOBA podría ser SEMINARIO CORDOBA). Se revisan a mano.
def token_previo(fila):
    complemento = fila["complemento"].split()
    return complemento[-1] if fila["barrio_oficial"] and complemento else None

medellin["token_previo"] = medellin.apply(token_previo, axis=1)
dudosos = medellin[
    medellin["token_previo"].notna()
    & ~medellin["token_previo"].isin(PALABRAS_VIVIENDA)
    & ~medellin["token_previo"].str.match(TOKEN_NUMERACION, na=False)
]
con_separador = medellin["Direccion de residencia"].astype(str).str.contains("-*-", regex=False) & con_texto
print("Con barrio oficial como sufijo:", int(medellin["barrio_oficial"].notna().sum()), "de", int(con_texto.sum()))
print("Ídem, solo registros con '-*-' (comparable con la exploración):",
      int((con_separador & medellin["barrio_oficial"].notna()).sum()), "de", int(con_separador.sum()))
print("Con un nombre que solo es de comuna:", int(es_comuna.sum()), "| comunas sin barrio homónimo:", sorted(comunas_sin_barrio))
print("Sufijos con token previo desconocido (revisión manual):", len(dudosos))
dudosos.groupby(["token_previo", "barrio_oficial"]).size().sort_values(ascending=False).to_frame("registros")

**Conclusión.** Ver las cifras de arriba y su resumen en `notebooks/hallazgos_geojson.md`. La medición preliminar de la
exploración (366 de 529) solo contaba registros con `-*-`; aquí se cuentan también los que no lo tienen, y la segunda
cifra de coincidencia, restringida a los registros con `-*-`, es la comparable. Si `original/` cambia, lo relevante es
la proporción. La lista de token previo desconocido es la que hay que revisar para detectar sufijos
que en realidad forman parte de un nombre más largo; muchos son el nombre de la comuna escrito antes del barrio
(`BELEN LOS ALPES`), que es correcto.

Dos ajustes respecto a la heurística descrita en el diseño, ambos hechos durante este análisis:
- Los nombres que solo existen como comuna o corregimiento (`SAN ANTONIO DE PRADO`, `MANRIQUE`...) se buscan junto con
  los barrios. Sin esto, `SAN ANTONIO DE PRADO` se asignaba al barrio `PRADO` (comuna La Candelaria) por su sufijo.
- Si no hay coincidencia, se reintenta sin los números finales (`FATIMA 4`). No se hace siempre porque algunos
  nombres oficiales acaban en número (`SAN JAVIER NO 1`).

## 6. Coincidencia con los nombres oficiales y resolución de repetidos

Tasas de coincidencia exacta (todo el resto es un nombre oficial) y por sufijo, sin alias y con alias, y aplicación de
la regla de nombres repetidos: si un nombre corresponde a varios polígonos, se elige el de `indicador_ur` igual a la
zona (`URBANO` → `U`, `RURAL` → `R`); si queda más de uno, o no hay zona, el barrio queda sin resolver.

In [ ]:
def cargar_alias(ruta=RUTA_ALIAS):
    """Lee la tabla de alias con codigo como texto. Devuelve un diccionario alias -> codigo (vacío si no existe)."""
    if not ruta.exists():
        return {}
    tabla = pd.read_csv(ruta, dtype=str, keep_default_na=False)
    return dict(zip(tabla["alias"], tabla["codigo"]))


alias = cargar_alias()
# Variantes de nombres de comuna decididas en la revisión de alias. No son alias de un barrio (D4): se cuentan como
# "solo comuna", candidatos al nivel COMUNA de ubicacion-por-niveles. Se buscan por sufijo, como los barrios, y el
# sufijo más largo gana: EL POBLADO sigue siendo el barrio y EL RODEO el barrio de Guayabal.
VARIANTES_COMUNA = {"POBLADO": "EL POBLADO", "RODEO": "BELEN", "RODEO ALTO": "BELEN", "RODEO SUR": "BELEN"}
nombres_comuna = comunas_sin_barrio | set(VARIANTES_COMUNA)
assert not set(VARIANTES_COMUNA) & nombres_oficiales
codigos_por_nombre = barrios.groupby("nombre_norm")["codigo"].agg(list).to_dict()
indicador_por_codigo = barrios.set_index("codigo")["indicador_ur"].to_dict()
LETRA_ZONA = {"URBANO": "U", "RURAL": "R"}


def resolver(nombre, zona, alias=alias):
    """Devuelve el codigo del polígono para un nombre normalizado y una zona, o None si no se puede resolver."""
    if nombre in alias:
        return alias[nombre]
    candidatos = codigos_por_nombre.get(nombre, [])
    if len(candidatos) == 1:
        return candidatos[0]
    if not candidatos or zona is None:
        return None
    por_zona = [c for c in candidatos if indicador_por_codigo[c] == LETRA_ZONA[zona]]
    return por_zona[0] if len(por_zona) == 1 else None


def comuna(codigo):
    return barrios.loc[barrios["codigo"] == codigo, "nombre_comuna"].item()


# Escenarios de la especificación "Resolución de nombres de barrio repetidos"
florida_urbano = resolver("LA FLORIDA", "URBANO", alias={})
florida_rural = resolver("LA FLORIDA", "RURAL", alias={})
assert comuna(florida_urbano) == "EL POBLADO" and indicador_por_codigo[florida_urbano] == "U"
assert comuna(florida_rural) == "SAN ANTONIO DE PRADO" and indicador_por_codigo[florida_rural] == "R"
assert resolver("SUBURBANO EL LLANO", "RURAL", alias={}) is None
assert resolver("LA FLORIDA", None, alias={}) is None
assert resolver("LAURELES", "RURAL", alias={}) == codigos_por_nombre["LAURELES"][0]  # nombre único, zona distinta
assert resolver("AREA DE EXPANSION PAJARITO", "URBANO", alias={}) is not None      # nombre único, sin indicador
print("Escenarios de resolución de repetidos: OK")

In [ ]:
def medir(nombres_validos, alias_activos, comunas):
    """Cuenta coincidencias exactas, por sufijo, resueltas y ambiguas sobre los registros de Medellín con texto."""
    datos = medellin[con_texto]
    exactas = datos["tokens"].map(lambda t: " ".join(t) in nombres_validos)
    encontrados = datos["tokens"].map(lambda t: buscar_nombre(t, nombres_validos | comunas)[0])
    solo_comuna = encontrados.isin(comunas)
    sufijos = encontrados.where(~solo_comuna)
    codigos = [resolver(n, z, alias_activos) if isinstance(n, str) else None for n, z in zip(sufijos, datos["zona"])]
    resueltos = pd.Series(codigos, index=datos.index, dtype=object)
    return {
        "con texto": len(datos),
        "coincidencia exacta": int(exactas.sum()),
        "coincidencia por sufijo": int(sufijos.notna().sum()),
        "resueltos a un polígono": int(resueltos.notna().sum()),
        "sin resolver por ambigüedad": int((sufijos.notna() & resueltos.isna()).sum()),
        "solo comuna (sin barrio)": int(solo_comuna.sum()),
    }, sufijos, encontrados.where(solo_comuna).map(lambda n: VARIANTES_COMUNA.get(n, n), na_action="ignore"), resueltos


metricas_sin_alias, _, _, _ = medir(nombres_oficiales, {}, comunas_sin_barrio)
metricas_con_alias, sufijo_final, comuna_final, codigo_declarado = medir(
    nombres_oficiales | set(alias), alias, nombres_comuna
)
medellin["barrio_declarado"] = sufijo_final
medellin["comuna_declarada"] = comuna_final
medellin["codigo_declarado"] = codigo_declarado

comparacion = pd.DataFrame({"sin alias": metricas_sin_alias, "con alias": metricas_con_alias})
comparacion["% sin alias"] = (100 * comparacion["sin alias"] / metricas_sin_alias["con texto"]).round(1)
comparacion["% con alias"] = (100 * comparacion["con alias"] / metricas_con_alias["con texto"]).round(1)
print("Alias cargados:", len(alias), "| variantes de comuna:", len(VARIANTES_COMUNA))
print("Registros que solo nombran una comuna, por comuna:", medellin["comuna_declarada"].value_counts().to_dict())
comparacion

In [ ]:
# Nombres repetidos que aparecen en los datos y cómo se resolvieron
ambiguos = medellin[medellin["barrio_declarado"].isin(nombres_repetidos)]
ambiguos.groupby(["barrio_declarado", ambiguos["zona"].fillna("(sin zona)")]).agg(
    registros=("codigo_declarado", "size"),
    resueltos=("codigo_declarado", lambda s: int(s.notna().sum())),
)

**Conclusión.** La coincidencia por sufijo recupera bastantes más barrios que la exacta, porque separa el complemento
(`APTO 301`, `UNIDAD ...`). La regla de zona resuelve casi todos los nombres repetidos; los que no resuelve quedan sin
resolver a propósito, en lugar de asignar un polígono al azar. Los registros que solo nombran una comuna sin barrio
homónimo no se asignan a ningún polígono: son los candidatos a un nivel COMUNA. La columna "con alias" muestra la
ganancia de la tabla de alias de la sección 7 y de las variantes de comuna (`POBLADO` como comuna El Poblado y los
sectores `RODEO` como comuna Belén), decididas en la misma revisión.

## 7. Candidatos para la tabla de alias

Textos de registros de Medellín que no coinciden con ningún nombre oficial ni alias, ordenados por frecuencia. Se quitan
los prefijos de vivienda (hasta la última palabra de vivienda y su numeración) y, para cada candidato, `difflib`
sugiere nombres oficiales parecidos comparando sus sufijos. Los nombres que solo existen como comuna ya se contaron en la
sección 6 y no deben ser alias de un barrio (D4); la columna `tipo` marca los que todavía aparezcan aquí.

In [ ]:
nombres_oficiales_lista = sorted(nombres_oficiales)
CORTES = PALABRAS_VIVIENDA | PALABRAS_VIA | {"KM", "VIA", "NIVEL"}


def limpiar_candidato(tokens):
    """Quita los números finales y todo lo anterior a la última palabra de vía o vivienda o al último token con dígitos.

    Los tokens con dígitos incluyen los pegados a una palabra (APTO301, TORRE2, KM5), que no son nombres de barrio.
    """
    tokens = list(tokens)
    while tokens and NUMERO.match(tokens[-1]):
        tokens.pop()
    cortes = [i for i, t in enumerate(tokens) if t in CORTES or re.search(r"\d", t)]
    if cortes:
        tokens = tokens[cortes[-1] + 1:]
    while tokens and TOKEN_NUMERACION.match(tokens[0]):
        tokens = tokens[1:]
    return " ".join(tokens)


def sugerencias(candidato, n=3):
    """Nombres oficiales parecidos a algún sufijo del candidato, con su similitud."""
    tokens = candidato.split()
    mejores = {}
    for k in range(1, min(len(tokens), 5) + 1):
        sufijo = " ".join(tokens[-k:])
        for nombre in difflib.get_close_matches(sufijo, nombres_oficiales_lista, n=n, cutoff=0.75):
            similitud = difflib.SequenceMatcher(None, sufijo, nombre).ratio()
            mejores[nombre] = max(mejores.get(nombre, 0), similitud)
    return [f"{nombre} ({similitud:.2f})" for nombre, similitud in sorted(mejores.items(), key=lambda kv: -kv[1])[:n]]


def tipo_candidato(candidato):
    tokens = candidato.split()
    sufijos = {" ".join(tokens[-k:]) for k in range(1, len(tokens) + 1)}
    return "comuna sin barrio" if sufijos & nombres_comuna else "posible alias"


sin_coincidencia = medellin[con_texto & medellin["barrio_declarado"].isna() & medellin["comuna_declarada"].isna()]
candidatos_limpios = sin_coincidencia["tokens"].map(limpiar_candidato)
solo_vivienda = int((candidatos_limpios == "").sum())
frecuencias = Counter(c for c in candidatos_limpios if c)
candidatos = pd.DataFrame(
    [(c, f, tipo_candidato(c), ", ".join(sugerencias(c))) for c, f in frecuencias.most_common()],
    columns=["candidato", "registros", "tipo", "sugerencias"],
)

assert not candidatos["candidato"].str.contains("-*-", regex=False).any()
assert not candidatos["candidato"].str.contains(r"\d").any()
assert not candidatos["candidato"].str.split().str[0].isin(PALABRAS_VIA).any()

print("Registros sin coincidencia:", len(sin_coincidencia), "| solo complemento de vivienda:", solo_vivienda)
print("Candidatos distintos:", len(candidatos), candidatos["tipo"].value_counts().to_dict())
candidatos

**Conclusión.** Esta lista es la entrada de la revisión manual: una persona decide cada alias y su motivo (`nota`) en
`data/alias_barrios.csv`. No se crean alias para los nombres de comuna sin barrio equivalente ni para prefijos de
vivienda; los primeros se cuentan como posible nivel COMUNA.

## 8. Línea base: puntos actuales de la API frente al barrio declarado

¿Los puntos que devolvió OpenCage (`coordenadas/coordenadas.xlsx`) caen en el barrio que declaró el estudiante? Se
excluyen los puntos en el centroide de Medellín, que no aportan información de barrio. Las distancias se miden en
EPSG:9377. Esta línea base servirá para evaluar la prueba A/B de `ubicacion-por-niveles`.

In [ ]:
coordenadas = pd.read_excel(RUTA_COORDENADAS, index_col=0)
# El índice de coordenadas.xlsx es el número de fila del archivo original (main.py lo conserva al filtrar).
mensaje = "coordenadas.xlsx no corresponde al archivo de original/: hay que volver a geocodificar"
assert coordenadas.index.isin(registros.index).all(), mensaje
assert (coordenadas["CODIGO"] == registros.loc[coordenadas.index, "CODIGO"]).all(), mensaje

con_punto = coordenadas["Latitud"].notna()
en_centroide = coordenadas["Latitud"].round(5).eq(CENTROIDE_MEDELLIN[0]) & coordenadas["Longitud"].round(5).eq(CENTROIDE_MEDELLIN[1])
evaluables = coordenadas[con_punto & ~en_centroide].join(medellin[["codigo_declarado"]], how="inner")
evaluables = evaluables[evaluables["codigo_declarado"].notna()]

print("Registros en coordenadas.xlsx:", len(coordenadas))
print("Registros de Medellín cubiertos por coordenadas.xlsx:", int(medellin.index.isin(coordenadas.index).sum()), "de",
      len(medellin))
print("Con punto:", int(con_punto.sum()), "| en el centroide de Medellín:", int(en_centroide.sum()))
print("Puntos fuera del centroide, de Medellín y con barrio declarado resuelto:", len(evaluables))

In [ ]:
puntos = gpd.GeoDataFrame(
    evaluables[["codigo_declarado"]],
    geometry=gpd.points_from_xy(evaluables["Longitud"], evaluables["Latitud"]),
    crs=CRS_WGS84,
).to_crs(CRS_METRICO)

poligonos = barrios.set_index("codigo")
union = gpd.sjoin(puntos, barrios[["codigo", "nombre_comuna", "geometry"]], how="left", predicate="within")
union = union[~union.index.duplicated()]  # un punto en el borde de dos polígonos se cuenta una vez

puntos["codigo_punto"] = union["codigo"]
puntos["en_barrio"] = puntos["codigo_punto"] == puntos["codigo_declarado"]
puntos["en_comuna"] = union["nombre_comuna"] == puntos["codigo_declarado"].map(poligonos["nombre_comuna"])
puntos["distancia_m"] = puntos.geometry.distance(
    gpd.GeoSeries(poligonos.loc[puntos["codigo_declarado"], "geometry"].values, index=puntos.index, crs=CRS_METRICO)
)

TOLERANCIAS = [0, 100, 250, 500, 1000, 2000]
curva = pd.DataFrame({
    "tolerancia_m": TOLERANCIAS,
    "puntos_aceptados": [int((puntos["distancia_m"] <= t).sum()) for t in TOLERANCIAS],
})
curva["porcentaje"] = (100 * curva["puntos_aceptados"] / len(puntos)).round(1)

print("Puntos evaluados:", len(puntos))
print("En el barrio declarado:", int(puntos["en_barrio"].sum()))
print("En la comuna del barrio declarado:", int(puntos["en_comuna"].sum()))
print("Fuera de Medellín (sin polígono):", int(puntos["codigo_punto"].isna().sum()))
print("Distancia al barrio declarado (m): mediana", round(float(puntos["distancia_m"].median())))
curva

**Conclusión.** Ver las cifras de arriba. En la exploración, con todos los registros geocodificados, solo 20 de 162
puntos caían en el barrio declarado y 59 en su comuna. Si `coordenadas.xlsx` solo cubre una parte de los registros de
Medellín (ver la segunda cifra de arriba), la línea base es parcial; las diferencias también se deben a los alias. La curva de tolerancias es la que orienta el umbral de aceptación del nivel
de la API en `ubicacion-por-niveles`.

## 9. Resumen y recomendaciones para `ubicacion-por-niveles`

Cifras agregadas de las secciones anteriores. Son las que se copian a `notebooks/hallazgos_geojson.md`.

In [ ]:
con_texto_total = metricas_con_alias["con texto"]
sin_resolver = con_texto_total - metricas_con_alias["resueltos a un polígono"] - metricas_con_alias["solo comuna (sin barrio)"]


def porcentaje(parte, total):
    return round(100 * parte / total, 1)


resumen = pd.DataFrame([
    ("GeoJSON", "polígonos válidos", len(barrios) - int((~barrios.is_valid).sum()), None),
    ("GeoJSON", "nombres repetidos tras normalizar", len(nombres_repetidos), None),
    ("GeoJSON", "centroides fuera de su polígono", int(centroide_fuera.sum()), None),
    ("Registros", "en original/", len(registros), None),
    ("Registros", "de Medellín", len(medellin), porcentaje(len(medellin), len(registros))),
    ("Medellín con texto", "total", con_texto_total, 100.0),
    ("Medellín con texto", "barrio resuelto sin alias", metricas_sin_alias["resueltos a un polígono"],
     porcentaje(metricas_sin_alias["resueltos a un polígono"], con_texto_total)),
    ("Medellín con texto", "barrio resuelto con alias", metricas_con_alias["resueltos a un polígono"],
     porcentaje(metricas_con_alias["resueltos a un polígono"], con_texto_total)),
    ("Medellín con texto", "solo comuna (posible nivel COMUNA)", metricas_con_alias["solo comuna (sin barrio)"],
     porcentaje(metricas_con_alias["solo comuna (sin barrio)"], con_texto_total)),
    ("Medellín con texto", "sin resolver (ni barrio ni comuna)", sin_resolver, porcentaje(sin_resolver, con_texto_total)),
    ("Medellín con texto", "sufijos con token previo desconocido", len(dudosos), porcentaje(len(dudosos), con_texto_total)),
    ("API (línea base)", "registros de Medellín cubiertos por coordenadas.xlsx",
     int(medellin.index.isin(coordenadas.index).sum()), porcentaje(int(medellin.index.isin(coordenadas.index).sum()), len(medellin))),
    ("API (línea base)", "puntos evaluados", len(puntos), 100.0),
    ("API (línea base)", "en el barrio declarado", int(puntos["en_barrio"].sum()), porcentaje(int(puntos["en_barrio"].sum()), len(puntos))),
    ("API (línea base)", "en la comuna del barrio declarado", int(puntos["en_comuna"].sum()), porcentaje(int(puntos["en_comuna"].sum()), len(puntos))),
], columns=["tema", "medida", "valor", "porcentaje"])
resumen

**Recomendaciones.**

1. **Punto cero del barrio:** `representative_point()` calculado en EPSG:9377 y transformado a WGS84 al final (sección 4).
   El centroide puede caer fuera del polígono.
2. **Extracción del barrio:** partir del texto original, no de `estandarizado/`; usar la normalización de la
   especificación, el sufijo más largo, la tabla de alias y la regla de zona para los nombres repetidos. Los nombres que
   solo designan una comuna se buscan junto con los barrios para que no se confundan con un barrio por su sufijo.
3. **Nivel COMUNA:** los registros que solo nombran una comuna (incluidas las variantes `POBLADO` y `RODEO`) no deben
   asignarse a un barrio. Conviene un nivel intermedio con un punto de la comuna, que puede obtenerse uniendo sus
   polígonos de barrio (`dissolve` por `nombre_comuna`) sin otra fuente.
4. **Tolerancia inicial para aceptar el punto de la API:** la curva de la sección 8 crece rápido hasta 250 m (puntos
   junto al borde del barrio) y despacio después, cuando empiezan a entrar barrios vecinos. Se propone empezar con 250 m,
   como parámetro configurable, y volver a calibrarla cuando `coordenadas.xlsx` cubra todos los registros, porque la línea
   base actual es parcial.
5. **Casos sin resolver:** se espera que quede sin barrio ni comuna en torno a la proporción de la tabla de arriba. Para
   ellos, el nivel siguiente es el punto del municipio. La sección 7 sirve para revisar los candidatos de cada periodo
   nuevo y ampliar la tabla de alias.